# VTT Meeting Transcript → Embedding-Ready Chunks Pipeline (Full Batch Draft)

**Objective:** Convert raw WebVTT meeting transcript files stored in a Unity Catalog Volume into structured, PII-governed chunks suitable for embedding and RAG retrieval. This notebook is a full-batch draft — every layer is a complete overwrite each run. A production version would swap Bronze ingestion for incremental Auto Loader and Silver/Gold writes for merge-based upserts, but the batch version is easier to reason about while learning the pattern.

**Why this notebook is useful for Generative AI Engineer exam prep:** it's a compact, working example of the *Data Preparation for Generative AI Applications* domain end-to-end — landing raw unstructured files, governing them (PII redaction, both deterministic and LLM-based), and chunking them for retrieval — plus hands-on use of Databricks **AI Functions** (`ai_mask`, `ai_classify`) from the *Application Development* domain, which call a foundation model directly from SQL/DataFrame code with no inference pipeline to stand up.

## Architecture (Medallion)

The medallion pattern (Bronze → Silver → Gold) is the standard way Databricks structures a pipeline: each layer is a Delta table with progressively more refinement, so consumers can pick the layer that matches their trust/latency needs (raw for reprocessing, silver for governed access, gold for serving).

| Layer | Table | Purpose |
|-------|-------|--------|
| **Bronze** | `dev.callagent_bronze.vtt_raw` | Raw VTT file content, one row per file — a faithful, unmodified copy of what landed in the Volume |
| **Silver** | `dev.callagent_silver.utterances` | Parsed transcript text: PII-redacted (regex + LLM), call type labelled |
| **Gold** | `dev.callagent_gold.transcript_chunks` | Fixed-size, overlapping chunks ready to feed an embedding model and Vector Search index |

Each table name follows Unity Catalog's three-level namespace, `catalog.schema.table` — there's no such thing as a four-level `catalog.database.layer.table` path. Each medallion layer gets its own **schema**, not a nested folder.

## Unity Catalog concepts this notebook relies on

- **Three-level namespace** — every table and volume is addressed as `catalog.schema.object`. This notebook uses one catalog (`dev`) with three schemas (`callagent_bronze`, `callagent_silver`, `callagent_gold`), one per medallion layer — a common pattern for separating governance, since you can grant access to `callagent_gold` without exposing raw, PII-laden bronze data.
- **Volumes** — `VOLUME_ROOT` points at a UC **Volume**, the governed way to read/write arbitrary files that aren't a managed Delta table (raw VTT files here, but the same mechanism holds model weights, images, or any blob). Volumes carry their own access controls, independent of the Delta tables built from them — the raw-vs-curated governance split this project's architecture is built around.
- **`CREATE CATALOG/SCHEMA IF NOT EXISTS`** is idempotent DDL, safe to re-run — it's what lets this notebook run start-to-finish from a clean workspace with no manual setup step.

In [ ]:
import re
import hashlib
import json
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ── Catalog & Schema ──────────────────────────────────────────────
CATALOG = "dev"
BRONZE_SCHEMA = "callagent_bronze"
SILVER_SCHEMA = "callagent_silver"
GOLD_SCHEMA   = "callagent_gold"

# ── Source Volume ─────────────────────────────────────────────────
VOLUME_ROOT = f"/Volumes/{CATALOG}/callagent/raw/teams_transcripts"

# ── Table Names ──────────────────────────────────────────────────
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.vtt_raw"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.utterances"
GOLD_TABLE   = f"{CATALOG}.{GOLD_SCHEMA}.transcript_chunks"

# ── Create schemas if needed ─────────────────────────────────────
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

print("✅ Configuration loaded")
print(f"   Source volume : {VOLUME_ROOT}")
print(f"   Bronze table  : {BRONZE_TABLE}")
print(f"   Silver table  : {SILVER_TABLE}")
print(f"   Gold table    : {GOLD_TABLE}")

## Bronze Layer: Raw VTT Ingestion

Reads `.vtt` files from the volume directory tree (partitioned by `call_date=YYYY-MM-DD`).
Each file becomes one row holding the **entire raw file content as a single string**, plus metadata extracted from its path and filename.

Two ideas worth calling out for the exam's *Data Preparation* domain:

- **Bronze is a copy, not a transformation.** Nothing here parses the VTT structure or touches PII — that's deliberate. If a downstream parsing rule changes, or a bug turns up in Silver, Silver/Gold can be rebuilt from Bronze without re-reading the source files. This is the medallion pattern's core value: expensive or fragile source reads happen once.
- **`_metadata.file_path`** is a Databricks-provided hidden column available on any file-based read — it gives the source path without a separate file-listing step, which is what the `call_date=` and `meeting_name` extraction below relies on via `regexp_extract`.
- **`call_id = sha2(file_path, 256)`** — hashing the file path gives a deterministic, stable primary key: the same file always produces the same `call_id` on re-ingestion, which is what makes the `mode("overwrite")` full-refresh pattern below safe to re-run repeatedly (idempotency).

In [ ]:
# ── Batch read all VTT files from the volume ─────────────────────
# Each file is read as a single row with `value` = full file text
raw_files = (
    spark.read.format("text")
    .option("wholetext", "true")
    .load(f"{VOLUME_ROOT}/**/*.vtt")
    .withColumnRenamed("value", "file_content")
    .withColumn("file_path", F.col("_metadata.file_path"))
)

# Extract metadata from partition path and file name
# Generate call_id from filename (deterministic hash of file path)
bronze_df = (
    raw_files
    .withColumn("call_date", F.regexp_extract(F.col("file_path"), r"call_date=(\d{4}-\d{2}-\d{2})", 1).cast("date"))
    # .withColumn("file_name", F.regexp_extract(F.col("file_path"), r"([^/]+)\.vtt$", 1))
    .withColumn("file_name", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.regexp_extract(F.col("file_path"), r"([^/]+)\.vtt$", 1), "%20", " "), "%5B", "["), "%5D", "]"))
    .withColumn("meeting_name", F.regexp_extract(F.col("file_name"), r"(.+)-20\d{6}_\d{6}-Meeting Recording$", 1))
    .withColumn("call_id", F.sha2(F.col("file_path"), 256))
    .withColumn("ingestion_ts", F.current_timestamp())
    .select(
        "call_id",
        "call_date",
        "file_path",
        "file_name",
        "meeting_name",
        "file_content",
        "ingestion_ts",
    )
)

print(f"Raw VTT files found: {bronze_df.count()}")
bronze_df.limit(10).display()

### A note on write modes

The `.write.mode("overwrite").option("overwriteSchema", "true")` pattern below fully replaces the table's data **and** allows the schema to change — appropriate here since this is a full-batch rebuild notebook. In a production/incremental pipeline you'd more likely see:
- **Auto Loader**, with `.trigger(availableNow=True)` or continuous streaming, to only process new files instead of re-reading everything;
- **`MERGE INTO`**, to upsert changed rows instead of overwriting the whole table.

Recognizing when each write pattern applies — full overwrite vs. incremental merge vs. streaming — is a recurring theme across the exam's Data Preparation and Assembling/Deploying domains.

In [ ]:
# ── Write bronze table with full refresh ──
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)
print(f"✅ Bronze table written: {BRONZE_TABLE}")

## Silver Layer: Structured, Governed Calls

Parses each call's raw WebVTT into a single flattened, newline-joined `"Speaker: text"` transcript — **one row per call**, not one row per utterance — then applies PII redaction and call classification. Output columns:

- `call_id`, `call_date`, `file_name`, `meeting_name` (carried from Bronze)
- `transcript_text` — the parsed, flattened transcript, no redaction yet
- `regex_redacted_text` — deterministic PII patterns masked (next cell)
- `ai_redacted_text` — `regex_redacted_text` further masked by the LLM-based `ai_mask()` function, falling back to `regex_redacted_text` if the AI Function call fails
- `pii_flagged`, `utterance_count`, `character_count` — governance/QA metadata
- `call_type` — one of `support`, `sales`, `internal`, `onboarding`, `other`, assigned by `ai_classify()`

**PII masking is deliberately two-stage, not just `ai_mask()` alone:**
1. **Regex first** — fast, free, fully deterministic, and catches predictable formats (emails, phone numbers, card-number-shaped digit groups). Runs entirely on the cluster, no external model call.
2. **`ai_mask()` second** — a Databricks **AI Function** that calls a foundation model to catch PII regex can't reliably pattern-match (a spoken date of birth, a full street address in free text). It costs tokens and adds latency, so it only runs on the text regex already reduced.

This "cheap deterministic pass first, expensive semantic pass second" layering is a general governance/cost pattern worth remembering for the exam — always check whether rules can handle part of a problem before reaching for an LLM call on every row.

**Call classification** via `ai_classify()` is a different AI Function — a zero-shot classifier against a fixed label set, useful anywhere you'd otherwise train and host a small classification model. Both `ai_mask()` and `ai_classify()` run as **batch inference over a Delta table** — no endpoint to deploy, no client SDK, just a SQL/DataFrame expression — which is exactly what the exam means by "AI Functions for batch inference."

The next few cells build this up in order: VTT parsing → regex PII patterns → a quick single-record test → an `ai_mask()` demo → the full Silver build.

In [ ]:
# ── VTT Parsing Logic ────────────────────────────────────────────
# WebVTT format:
#   WEBVTT\n\n
#   00:00:00.000 --> 00:00:14.143\n
#   <v Speaker Name>Utterance text</v>\n\n
#   00:00:14.843 --> 00:00:33.700\n
#   <v Speaker Name>More text</v>\n

TS_PATTERN = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})\.(\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})\.(\d{3})"
)
SPEAKER_PATTERN = re.compile(r"<v\s+([^>]+)>(.*?)</v>", re.DOTALL)

# ── PII Redaction Patterns ───────────────────────────────────────
PII_PATTERNS = [
    # Email addresses
    re.compile(r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b"),
    # US phone numbers ((XXX) XXX-XXXX or XXX-XXX-XXXX)
    re.compile(r"\(?\b\d{3}\)?[\s.\-]?\d{3}[\s.\-]?\d{4}\b"),
    # Sensitive numbers (groups of 4 digits like the end of payment cards or account ids)
    re.compile(r"\b(?:\d{4}[\s\-]?){3}\d{4}\b"),
    # IPv4 addresses
    re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
    # ZIP codes (5 digits, but avoid matching within longer numbers)
    re.compile(r"\b\d{5}(?:\-\d{4})?\b"),
]

def _redact_pii(text: str) -> str:
    """Mask common PII patterns in text with placeholders."""
    if not text:
        return text
    redacted = text
    for pattern in PII_PATTERNS:
        redacted = pattern.sub("[MASKED]", redacted)
    return redacted


def _ts_to_seconds(h: str, m: str, s: str, ms: str) -> float:
    """Convert timestamp components to total seconds."""
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0


def parse_vtt(file_content: str) -> dict:
    """
    Parse a WebVTT transcript string into a single flat, single-line,
    speaker-tagged transcript for one call.

    Returns a dict:
        {
            "transcript_text": "Speaker: text Speaker: text ...",
            "redacted_transcript_text": "<PII-masked version of the above>",
            "pii_flagged": bool,
            "utterance_count": int,
        }
    """
    empty = {
        "transcript_text": "",
        "redacted_transcript_text": "",
        "pii_flagged": False,
        "utterance_count": 0,
    }
    if not file_content or not file_content.strip():
        return empty

    blocks = re.split(r"\n\s*\n", file_content.strip())
    utterance_lines = []

    for block in blocks:
        block = block.strip()
        if not block or block.startswith("WEBVTT"):
            continue

        ts_match = TS_PATTERN.search(block)
        if not ts_match:
            continue

        text_part = block[ts_match.end():].strip()

        speaker_match = SPEAKER_PATTERN.search(text_part)
        if speaker_match:
            speaker = speaker_match.group(1).strip()
            utterance_text = speaker_match.group(2).strip()
        else:
            speaker = "Unknown"
            utterance_text = re.sub(r"<[^>]+>", "", text_part).strip()

        # ── Sanitation, applied BEFORE joining and BEFORE PII masking ──
        # Collapse any internal newlines/repeated whitespace within a
        # single utterance so each speaker turn is itself a clean
        # single line before it becomes one line of the transcript.
        utterance_text = re.sub(r"\s+", " ", utterance_text).strip()

        if utterance_text:
            utterance_lines.append(f"{speaker}: {utterance_text}")

    if not utterance_lines:
        return empty

    # Newline join — one speaker turn per line.
    transcript_text = "\n".join(utterance_lines)

    regex_redacted_text = _redact_pii(transcript_text)
    pii_flagged = regex_redacted_text != transcript_text

    return {
        "transcript_text": transcript_text,
        "regex_redacted_text": regex_redacted_text,
        "pii_flagged": pii_flagged,
        "utterance_count": len(utterance_lines),
    }


# Register as a Spark UDF returning a single struct (one row per call)
CALL_SCHEMA = T.StructType([
    T.StructField("transcript_text",      T.StringType(),  False),
    T.StructField("regex_redacted_text",  T.StringType(),  False),
    T.StructField("pii_flagged",          T.BooleanType(), False),
    T.StructField("utterance_count",      T.IntegerType(), False),
])

parse_vtt_udf = F.udf(parse_vtt, CALL_SCHEMA)

print("✅ VTT parser registered as UDF (one row per call, flat single-line transcript text, regex-based PII redaction)")

Before running the parser across the full table, sanity-check it on one known record. This is a cheap habit that catches malformed regex or off-by-one parsing bugs before they're buried inside a UDF running over every row — much easier to debug one string than 32 at once in a `display()` grid.

In [ ]:
# PII masking test on sample file
test_meeting = "[Cascadence Support] Ferro & Vance Logistics — Billing Update"
test_content = spark.table(BRONZE_TABLE).where(f"meeting_name='{test_meeting}'").collect()[0]["file_content"]

test_parsed = parse_vtt(test_content)
print(f"{test_parsed['transcript_text']}\n -> {test_parsed['regex_redacted_text']}\n")

Some PII remains after regex-based masking — partial credit card numbers, dates of birth, and full addresses are all *semantically* recognizable but not reliably pattern-matchable (a birthday can be written a dozen different ways in natural language). For PII like this, Databricks' built-in [`ai_mask()`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_mask) function calls a foundation model to find and redact it based on a natural-language label, not a fixed pattern.

`ai_mask(content, array(label1, label2, ...))` — each label is a short natural-language description of what to find (e.g. `'birthday'`, not a regex). Under the hood this routes through a Databricks-hosted foundation model endpoint, governed the same way any Model Serving endpoint is (rate limits, cost, access control via Unity Catalog) — worth remembering that every `ai_mask()` call is a real inference call, not a free local operation, which is why it runs as the *second* pass here, only on text that regex has already reduced.

In [ ]:
# LLM-based PII masking using ai_mask()
# ai_mask(content, array('label1','label2',...)) uses a foundation model
# to detect and mask named entities, replacing matched text with [MASKED].

# Address remaining PII from same test case
test_content_df = spark.createDataFrame([test_parsed])

mask_labels = ['birthday', 'credit_card_numbers (last 4 digits)', 'physical_address (not references to addresses)']
labels_sql = ", ".join(f"'{l}'" for l in mask_labels)
test_ai_masked_df = test_content_df.withColumn(
    "ai_redacted_text",
    F.expr(f"ai_mask(regex_redacted_text, array({labels_sql}))")
)
test_ai_masked_df[['regex_redacted_text', 'ai_redacted_text']].display()

A few Databricks-specific patterns in the full Silver build below, worth recognizing for the exam:

- **`ai_classify(text, '[label1, label2, ...]', MAP('version', '2.1'))`** returns a **`VARIANT`** — a semi-structured type that can hold nested JSON-like data in a single column without a fixed schema. `variant_get(col, '$.response[0].value', 'string')` then pulls out just the label string. VARIANT is how AI Functions return rich, model-specific metadata (confidence, raw response, errors) without forcing every caller into a rigid struct schema.
- **`F.coalesce(ai_mask(...), regex_redacted_text)`** — if the `ai_mask()` call fails or returns null (a transient endpoint error, a rate limit), the pipeline falls back to the already-regex-redacted text rather than failing the whole job or silently passing through unredacted PII. This fail-safe-not-fail-open pattern matters for any governance-sensitive step that depends on an external model call.

In [ ]:
# ── Silver: parse VTT into one row per call with PII masked using both regex and ai_mask() ───
# - PII masking using regex patterns followed by ai_mask()
# - call labelling using ai_classify()

# PII Labels for ai_mask()
mask_labels = ['birthday', 'credit_card_numbers (last 4 digits)', 'physical_address (not references to addresses)']
labels_sql = ", ".join(f"'{l}'" for l in mask_labels)

# Call Type Labels (closed set for ai_classify)
call_type_labels = ['support', 'sales', 'internal', 'onboarding', 'other']
call_type_labels_sql = ", ".join(f'"{l}"' for l in call_type_labels)

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("parsed", parse_vtt_udf(F.col("file_content")))
    .select(
        "call_id",
        "call_date",
        "file_name",
        "meeting_name",
        F.col("parsed.transcript_text").alias("transcript_text"),
        F.col("parsed.regex_redacted_text").alias("regex_redacted_text"),
        F.col("parsed.pii_flagged").alias("pii_flagged"),
        F.col("parsed.utterance_count").alias("utterance_count"),
    )
    .withColumn("character_count", F.length(F.col("transcript_text")))
    # use LLM-based approach to mask additional PII, fallback to regex-redacted text if LLM fails
    .withColumn(
        "ai_redacted_text", 
        F.coalesce(
            F.expr(f"ai_mask(regex_redacted_text, array({labels_sql}))"),
            F.col("regex_redacted_text")
        )
    )
    # ai_classify() assigns exactly one label from call_type_labels based on full redacted call transcript
    # output is a VARIANT type containing the label, metadata, and error_message
    .withColumn(
        "call_type_raw",
        F.expr(
            f"""
                ai_classify(
                    concat(
                        ai_redacted_text,
                        ' | Meeting Name: ', meeting_name
                    ),
                    '[{call_type_labels_sql}]',
                    MAP('version', '2.1')
                )
            """
        )
    )
    # retrieve the label from the ai_classify() response
    .withColumn(
        "call_type",
        F.expr("variant_get(call_type_raw, '$.response[0].value', 'string')")
    )
    .drop("call_type_raw")
)
silver_df.limit(10).display()

In [ ]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)
print(f"✅ Silver table written: {SILVER_TABLE}")
print(f"   Total calls: {spark.table(SILVER_TABLE).count()}")
spark.table(SILVER_TABLE).limit(10).display()

## Gold Layer: Redacted Chunks for Embedding

To prepare this table for a Databricks **Vector Search** index, the fully redacted transcript text needs to be split into smaller **chunks** — the unit that actually gets embedded and retrieved.

Two reasons chunking matters, both core to the exam's RAG/Application Development content:

1. **Embedding models have a context window.** Text longer than the model's max input gets silently truncated (or errors, depending on the model), losing whatever fell past the limit — chunking keeps every piece well under that ceiling.
2. **Retrieval quality, not just fit.** Even when a full call transcript *would* technically fit in one embedding call, a single embedding vector for a 20-minute conversation is too diluted a summary to match a specific user question well. Smaller, focused chunks let the retriever return the specific passage relevant to a query instead of an entire, mostly-irrelevant transcript.

This notebook uses **`chunk_size=500`, `chunk_overlap=100`** characters via LangChain's `RecursiveCharacterTextSplitter` (next two cells). Overlap exists so a sentence or idea split across a chunk boundary still appears in full in at least one chunk — without it, a boundary landing mid-thought can leave both halves individually un-retrievable for a question about that thought. Recursive splitting tries the highest-priority separator first (here: newline, i.e. speaker-turn boundaries) and only falls back to a lower-priority one (sentence punctuation, then whitespace) if a piece is still too long — keeping chunks aligned to natural conversation boundaries instead of cutting mid-word at a fixed character count, which is the main advantage over naive fixed-size chunking.

`langchain-text-splitters` isn't part of the Databricks Runtime, so it needs `%pip install` — and `dbutils.library.restartPython()` immediately after is required, not optional. The Python process needs a fresh interpreter to pick up a newly installed package, since import state is cached per-process. Skipping the restart is a common source of a `ModuleNotFoundError` right after a `%pip install` that appeared to succeed.

In [ ]:
%pip install -q langchain-text-splitters
dbutils.library.restartPython()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500
chunk_overlap = 100

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n", ".", "!", "?", " "],
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    is_separator_regex=False,
)


def chunk_text(text: str, call_id: str) -> list:
    """
    Recursive chunking via LangChain's RecursiveCharacterTextSplitter.
    Separator priority: "\n" (speaker-turn boundary) first, then "."
    (sentence boundary), then "," (clause boundary) as fallback — falls
    back further to a hard character cut internally if none apply.

    Call-boundary safety: this function only ever receives ONE call's
    transcript text at a time (applied per-row on a one-row-per-call
    table), so a chunk can never span two calls by construction.

    Returns: list of chunk dicts matching CHUNK_SCHEMA
    """
    if not text:
        return []

    pieces = text_splitter.split_text(text)

    chunks = []
    for i, piece in enumerate(pieces):
        chunk_id = hashlib.sha256(f"{call_id}:{i}".encode("utf-8")).hexdigest()
        chunks.append({
            "chunk_id":    chunk_id,
            "chunk_index": i,
            "chunk_text":  piece.strip(),
        })
    return chunks


chunk_schema = T.ArrayType(T.StructType([
    T.StructField("chunk_id",     T.StringType(),  False),
    T.StructField("chunk_index",  T.IntegerType(), False),
    T.StructField("chunk_text",   T.StringType(),  False),
]))

chunk_text_udf = F.udf(
    lambda text, call_id: chunk_text(text, call_id),
    chunk_schema,
)

all_chunks_df = (
    spark.table(SILVER_TABLE)
    .withColumn("chunks", chunk_text_udf(F.col("ai_redacted_text"), F.col("call_id")))
    .select(
        "call_id",
        "call_date",
        "call_type",
        F.explode("chunks").alias("chunk"),
    )
    .select(
        F.col("chunk.chunk_id").alias("chunk_id"),
        F.col("chunk.chunk_index").alias("chunk_index"),
        "call_id",
        "call_date",
        "call_type",
        F.col("chunk.chunk_text").alias("chunk_text"),
    )
)

print("✅ Chunking transformation complete — ready to write to gold")
all_chunks_df.limit(10).display()

In [ ]:
# Write chunk data to gold table
(
    all_chunks_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"✅ Gold table written: {GOLD_TABLE}")
print(f"   Total chunks: {spark.table(GOLD_TABLE).count()}")
spark.table(GOLD_TABLE).limit(10).display()

## What's next

This notebook covers the exam's Data Preparation domain end-to-end: raw ingestion → governed PII redaction → chunking. `dev.callagent_gold.transcript_chunks` is now ready to feed:

1. **An embedding model** — a Databricks Foundation Model API embedding endpoint, or a hosted open model — run over `chunk_text`. This notebook stops short of that step and leaves the embedding column for a follow-on notebook.
2. **A Databricks Vector Search index**, built on top of the embedded gold table, enabling semantic search over chunks.
3. **A RAG chain**, which retrieves relevant chunks for a user query and passes them to an LLM to ground its answer — the exam's Application Development domain covers prompt engineering and chain assembly on top of exactly this kind of retrieval step.
4. **Evaluation**, scoring retrieval and generation quality against a held-out question set (e.g. MLflow's LLM evaluation harness) — the exam's Evaluation and Monitoring domain.